# MoE-MedIR — Google Colab Training Notebook

> **Paper**: Bridging the Intra-Modal Heterogeneity Gap: Mixture-of-Experts for Multi-Modality Medical Image Retrieval  
> **Conference**: ICARCV 2026

**Setup**: Runtime → Change runtime type → T4 GPU  
Run cells top-to-bottom.

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')

## Cell 2 — Install Packages

In [ ]:
%%capture
!pip install medmnist open_clip_torch timm pytorch-metric-learning scikit-learn matplotlib seaborn
print('Done.')

## Cell 3 — Clone Project từ GitHub

**Bước 1**: Tạo repo trên GitHub rồi đổi `REPO_URL` bên dưới.  
**Bước 2**: Mỗi lần mở lại Colab, cell này sẽ `git pull` để lấy code mới nhất.

In [ ]:
import os

REPO_URL     = 'https://github.com/YOUR_USERNAME/moe_medir.git'  # <-- đổi
PROJECT_ROOT = '/content/moe_medir'

if os.path.exists(PROJECT_ROOT):
    !git -C {PROJECT_ROOT} pull
else:
    !git clone {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
!git log --oneline -3

## Cell 3.5 — ⚙️ Config Editor

**Chỉnh sửa tại đây** trước khi chạy train. Sau khi chạy cell này, config.py sẽ được cập nhật tự động.

| Tham số | Các lựa chọn |
|---|---|
| `BACKBONE` | `clip_vitb32` · `biomedclip` · `dinov2_vitb14` |
| `FEATURE_MODE` | `concat` (CLS+Patch, 1536-dim) · `cls` (CLS only, 768-dim) |
| `ROUTING_MODE` | `token_choice` · `expert_choice` |
| `TOP_K` | 2 (sparse, specialised) · 4 (dense) |
| `EMBED_DIM` | 128 · 256 |

In [ ]:
import re, sys

# ════════════════════════════════════════════════════════════════
#  ✏️  CHỈNH SỬA CONFIG TẠI ĐÂY
# ════════════════════════════════════════════════════════════════

BACKBONE     = "clip_vitb32"   # clip_vitb32 | biomedclip | dinov2_vitb14
FEATURE_MODE = "concat"        # concat (CLS+Patch 1536) | cls (CLS only 768)
ROUTING_MODE = "token_choice"  # token_choice | expert_choice
TOP_K        = 2               # 2 (sparse) | 4 (dense)
EMBED_DIM    = 128             # 128 | 256
NUM_EXPERTS  = 8               # số lượng experts
BATCH_SIZE   = 256
EPOCHS       = 50
LR           = 1e-4

# ════════════════════════════════════════════════════════════════
#  🔧  Tự động ghi vào config.py — KHÔNG cần sửa bên dưới
# ════════════════════════════════════════════════════════════════

def _patch(content, field, value):
    if isinstance(value, str):
        pattern     = rf'([ \t]+{field}\s*:\s*\w+\s*=\s*)["\'].*?["\']'
        replacement = rf'\g<1>"{value}"'
    else:
        pattern     = rf'([ \t]+{field}\s*:\s*[\w\[\]]+\s*=\s*)[\d.e+\-]+'
        replacement = rf'\g<1>{value}'
    new, n = re.subn(pattern, replacement, content)
    if n == 0:
        print(f"  [warn] field '{field}' not found in config.py")
    return new

with open('config.py', 'r') as f:
    cfg_text = f.read()

for field, val in [
    ('backbone',     BACKBONE),
    ('feature_mode', FEATURE_MODE),
    ('routing_mode', ROUTING_MODE),
    ('top_k',        TOP_K),
    ('embed_dim',    EMBED_DIM),
    ('num_experts',  NUM_EXPERTS),
    ('batch_size',   BATCH_SIZE),
    ('epochs',       EPOCHS),
    ('lr',           LR),
]:
    cfg_text = _patch(cfg_text, field, val)

with open('config.py', 'w') as f:
    f.write(cfg_text)

if 'config' in sys.modules:
    import importlib, config as _cfg_mod
    importlib.reload(_cfg_mod)
    from config import CFG
else:
    from config import CFG

print("=" * 55)
print("  Config hiện tại")
print("=" * 55)
print(f"  backbone     : {CFG.backbone}")
print(f"  feature_mode : {CFG.feature_mode}  →  feature_dim={CFG.feature_dim}")
print(f"  feature_dir  : {CFG.feature_dir}")
print(f"  routing_mode : {CFG.routing_mode}")
print(f"  top_k        : {CFG.top_k}  /  num_experts={CFG.num_experts}")
print(f"  embed_dim    : {CFG.embed_dim}")
print(f"  batch_size   : {CFG.batch_size}  epochs={CFG.epochs}  lr={CFG.lr}")
print("=" * 55)
print("✅  config.py đã được cập nhật.")

## Cell 4 — Mount Google Drive (lưu results)

Optional — backup checkpoint và results về Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_BACKUP = '/content/drive/MyDrive/moe_medir_results'
import os; os.makedirs(RESULTS_BACKUP, exist_ok=True)
print('Drive mounted. Backup path:', RESULTS_BACKUP)

## Cell 5 — Sanity Check

Kiểm tra imports + config + model **trước khi download data**.

In [ ]:
import os, sys
from config import CFG

if os.path.exists(os.path.join(CFG.feature_dir, 'pathmnist_train_feat.npy')):
    print(f'Features đã có tại {CFG.feature_dir} — skipping.')
    print(f'Zip: data/features_{CFG.backbone}.zip')
else:
    !python extract_features.py
    # Sau extract, zip được tạo tự động tại data/features_{backbone}.zip

# Download zip về máy local
from google.colab import files
zip_path = f'data/features_{CFG.backbone}.zip'
if os.path.exists(zip_path):
    print(f'\nDownload zip ({os.path.getsize(zip_path)/1e6:.1f} MB)?')
    # Bỏ comment dòng dưới để tải về:
    # files.download(zip_path)

## Cell 6 — Extract Features

Download MedMNIST 28px (~1.2GB) + extract 1024-dim features. Chạy 1 lần (~8 phút).

In [ ]:
import os
if os.path.exists('data/features/pathmnist_train_feat.npy'):
    print('Features already extracted — skipping.')
else:
    !python extract_features.py

## Cell 7 — Zero-Shot Baselines

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/zeroshot.py'], check=True)

## Cell 8 — HashNet Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/hashnet.py'], check=True)

## Cell 9 — Linear Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'linear', '--epochs', '50'], check=True)
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'linear'], check=True)

## Cell 10 — MLP Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'mlp', '--epochs', '50'], check=True)
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'mlp'], check=True)

## Cell 11 — MoE-MedIR (Main Model)

~15 phút trên T4. Log `avg mAP@R` mỗi 5 epochs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'moe', '--epochs', '50'], check=True)

## Cell 12 — Full Test Evaluation

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'moe'], check=True)

## Cell 13 — Compile Comparison Table

In [ ]:
import subprocess, sys, pandas as pd
subprocess.run([sys.executable, 'baselines/compile_table.py'], check=True)
df = pd.read_csv('results/table_main.csv')
print(df.to_string(index=False))

## Cell 14 — Print LaTeX Table

In [ ]:
with open('results/table_main.tex') as f:
    print(f.read())

## Cell 15 — Expert Routing Heatmap

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'analysis/routing_heatmap.py'], check=True)

## Cell 16 — t-SNE Visualisation

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'analysis/tsne_plot.py', '--model', 'moe', '--max_samples', '400'], check=True)

## Cell 17 — Copy Results to Google Drive

In [ ]:
import shutil
shutil.copytree('results', RESULTS_BACKUP, dirs_exist_ok=True)
print('Results copied to', RESULTS_BACKUP)